In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("price_SPY.csv")
print(df.head())

         Date       Close        High         Low        Open     Volume
0  2024-01-29  479.418243  479.564649  475.417179  475.963666   61322800
1  2024-01-30  479.047455  479.759825  478.286243  478.725399   58618400
2  2024-01-31  471.230652  477.281061  471.211115  476.832166  126011100
3  2024-02-01  477.398193  477.427468  472.128443  472.938436   91891600
4  2024-02-02  482.423981  484.082951  477.495792  477.837355   99228200


In [ ]:

def detect_volatility_periods(
    series: pd.Series,
    *,
    input_is_returns: bool = False,
    windows: list[int] | None = None,
    max_window: int = 63,                 # ~3 mois en jours ouvrés (21*3)
    annualize: bool = True,
    periods_per_year: int = 252,          # 252 (jours ouvrés) ou 365
    threshold: float | None = None,       # si None => vol globale multi-échelle
    threshold_mode: str = "global",       # "global" | "quantile"
    threshold_quantile: float = 0.90,     # si mode="quantile"
    min_period_len: int = 1,              # longueur min (en jours) d’une période
    join_gap: int = 0,                    # combler petites coupures (0 = aucune)
    center: bool = False,
    plot: bool = True,
    plot_series: bool = True,
):
    """
    Détecte des périodes de forte volatilité via volatilités roulantes multi-horizons.

    Paramètres principaux
    - series: index datetime, valeurs prix ou rendements
    - input_is_returns: True si series = rendements déjà
    - windows: liste d'horizons (en "pas" de la série). Si None -> 1..max_window
    - threshold:
        * None: par défaut = volatilité globale (sur toute la période) de la mesure multi-échelle
        * float: seuil absolu (dans l’unité de vol utilisée)
    - threshold_mode:
        * "global": seuil par défaut = moyenne de la vol multi-échelle sur toute la période
        * "quantile": seuil par défaut = quantile de la vol multi-échelle
    - join_gap: si >0, comble jusqu’à join_gap jours non-volatils entre deux périodes volatiles
    """

    s = series.dropna().copy()
    if not isinstance(s.index, pd.DatetimeIndex):
        raise ValueError("series doit avoir un DatetimeIndex.")

    # 1) Rendements
    if input_is_returns:
        r = s.astype(float)
    else:
        # log-returns (robuste)
        r = np.log(s.astype(float)).diff().dropna()

    if len(r) < 5:
        raise ValueError("Série trop courte après calcul des rendements.")

    # 2) Horizons (fenêtres)
    if windows is None:
        windows = list(range(1, max_window + 1))
    windows = sorted(set(int(w) for w in windows if w >= 1))
    if not windows:
        raise ValueError("windows doit contenir au moins une fenêtre >= 1.")

    # 3) Volatilités roulantes multi-horizons
    vol_df = pd.DataFrame(index=r.index)

    # 1-day "vol proxy" via absolute returns
    vol_1d = r.abs()
    if annualize:
        vol_1d = vol_1d * np.sqrt(periods_per_year)
    vol_df[1] = vol_1d

    # rolling vols for windows >= 2
    for w in windows:
        if w < 2:
            continue
        vol_w = r.rolling(window=w, min_periods=w, center=center).std()
        if annualize:
            vol_w = vol_w * np.sqrt(periods_per_year)
        vol_df[w] = vol_w


    # 4) Seuil (par défaut)
    if threshold is None:
        if threshold_mode == "global":
            thr = float(vol_ms.mean(skipna=True))
        elif threshold_mode == "quantile":
            thr = float(vol_ms.quantile(threshold_quantile))
        else:
            raise ValueError("threshold_mode doit être 'global' ou 'quantile'.")
    else:
        thr = float(threshold)

    # 5) Flag volatilité
    flag = (vol_ms > thr).fillna(False)

    # Option : combler petites coupures (join_gap)
    if join_gap > 0 and flag.any():
        # on comble les runs False de longueur <= join_gap entre deux True
        x = flag.astype(int)
        # repère les blocs
        grp = (x.diff().fillna(0) != 0).cumsum()
        blocks = pd.DataFrame({"x": x, "grp": grp}).groupby("grp")["x"].agg(["first", "size"])
        # indices de chaque bloc
        grp_first_idx = pd.Series(flag.index).groupby(grp).first()
        grp_last_idx = pd.Series(flag.index).groupby(grp).last()

        for g, row in blocks.iterrows():
            if row["first"] == 0 and row["size"] <= join_gap:
                # bloc 0 (False) petit : comble si entouré par True
                prev_g = g - 1
                next_g = g + 1
                if prev_g in blocks.index and next_g in blocks.index:
                    if blocks.loc[prev_g, "first"] == 1 and blocks.loc[next_g, "first"] == 1:
                        start = grp_first_idx.loc[g]
                        end = grp_last_idx.loc[g]
                        flag.loc[start:end] = True

    # 6) Extraire périodes (runs True)
    periods = []
    if flag.any():
        change = flag.ne(flag.shift(1)).cumsum()
        for _, chunk in flag.groupby(change):
            if chunk.iloc[0] is True:
                start = chunk.index[0]
                end = chunk.index[-1]
                length = (chunk.index.size)
                if length >= min_period_len:
                    periods.append((start, end))

    periods_df = pd.DataFrame(periods, columns=["start", "end"])
    if not periods_df.empty:
        periods_df["length_obs"] = (periods_df["end"] - periods_df["start"]).dt.days + 1

    # 7) Plot
    fig = None
    if plot:
        fig, ax = plt.subplots(figsize=(12, 5))

        # série (prix ou rendements) en fond
        if plot_series:
            if input_is_returns:
                ax.plot(r.index, r.values, linewidth=1)
                ax.set_ylabel("Returns")
            else:
                ax.plot(s.index, s.values, linewidth=1)
                ax.set_ylabel("Series")

        # seconde échelle pour la volatilité multi-échelle
        ax2 = ax.twinx()
        ax2.plot(vol_ms.index, vol_ms.values, linewidth=1)
        ax2.axhline(thr, linestyle="--", linewidth=1)
        ax2.set_ylabel("Volatility (multi-scale max)")

        # bandes pour périodes volatiles
        for (start, end) in periods:
            ax.axvspan(start, end, alpha=0.2)

        ax.set_title("Detected high-volatility periods")
        ax.grid(True, alpha=0.2)
        plt.tight_layout()

    return {
        "periods": periods_df,     # dates de début/fin
        "threshold": thr,
        "vol_ms": vol_ms,          # volatilité multi-échelle
        "vol_by_window": vol_df,   # vols par horizon
        "flag": flag,              # booléen par date
        "figure": fig
    }


In [7]:
df["Date"] = pd.to_datetime(df["Date"])
s = pd.Series(df["Close"].values, index=df["Date"])

res = detect_volatility_periods(
    s,
    input_is_returns=False,   # True si tu donnes déjà des returns
    max_window=126,           # jusqu’à ~6 mois (jours ouvrés)
    threshold=None,           # seuil par défaut
    threshold_mode="quantile",
    threshold_quantile=0.90,  # “top 10%” des vols
    join_gap=2,               # comble 1-2 jours de “faux calme”
    min_period_len=2,         # ignore les épisodes d’1 jour
    plot=True
)

print(res["threshold"])
print(res["periods"])        # DataFrame start/end
plt.show()


ValueError: min_periods 2 must be <= window 1